<div style="display:flex;align-items:center"><img src="recursos/logos/upc.png" width="230"><img src="recursos/logos/caixabank.png" width="155" style="margin-left:20px"><span style="margin-left:auto;text-align:right"><b>CaixaBank · Advanced Analytics Program</b><br>Python profesional para analítica avanzada</span></div>

# Proyecto profesional y cierre

**Duración:** 2 horas · Caso: entrega profesional del informe de operaciones

El informe ya funciona. El reto final es convertirlo en un proyecto que otra persona pueda instalar, configurar, ejecutar, verificar y ampliar sin depender de nuestro notebook.

## Resultado esperado y recorrido

La entrega debe generar `data/processed/informe_operaciones.csv` y conservar un contrato verificable. El recorrido reproduce los tres bloques del proyecto original:

1. **Repositorio reproducible:** estructura, entorno, dependencias y configuración.
2. **Código verificable:** tests, regresión, Ruff, Black, mypy y packaging.
3. **Producto operable:** CLI, Makefile, API, Git, CI/CD y documentación.

Cada concepto se demuestra sobre el mismo caso antes de aparecer en un ejercicio.

# Primera parte — del análisis local al repositorio reproducible

## 1. Estructura y responsabilidades

Una carpeta profesional separa código, configuración, datos, pruebas y documentación. La estructura no garantiza calidad, pero permite localizar responsabilidades y automatizar comprobaciones.

In [ ]:
from pathlib import Path
import json
import os
import sys
import tomllib

import pandas as pd
from pandas.testing import assert_frame_equal

RAIZ = Path('plantilla_informe_operaciones')
SOLUCION = Path('solucion_informe_operaciones')
REQUERIDOS = {
    'pyproject.toml', 'README.md', 'config/config.json',
    'src/informe_operaciones/config.py',
    'src/informe_operaciones/validacion.py',
    'src/informe_operaciones/pipeline.py',
    'src/informe_operaciones/cli.py',
    'tests/test_pipeline.py',
}

presentes = {str(r.relative_to(RAIZ)) for r in RAIZ.rglob('*') if r.is_file()}
ausentes = sorted(REQUERIDOS.difference(presentes))
print('Estructura completa:', not ausentes)
for ruta in sorted(presentes):
    print(ruta)

## 2. Entorno virtual, dependencias y paquete editable

Un entorno virtual aísla dependencias. `pyproject.toml` declara las dependencias necesarias; `pip freeze` describe todo lo instalado en un entorno concreto y no sustituye esa declaración. La instalación editable conecta `src/` con el entorno durante el desarrollo.

```bash
python -m venv .venv
# macOS/Linux: source .venv/bin/activate
# PowerShell: .\.venv\Scripts\Activate.ps1
python -m pip install -e ".[dev]"
```

In [ ]:
print('Python:', sys.version.split()[0])
print('Intérprete:', sys.executable)
print('Dentro de un venv:', sys.prefix != getattr(sys, 'base_prefix', sys.prefix))

with (RAIZ / 'pyproject.toml').open('rb') as fichero:
    pyproject = tomllib.load(fichero)

print('Paquete:', pyproject['project']['name'])
print('Dependencias de ejecución:', pyproject['project']['dependencies'])
print('Extras:', pyproject['project']['optional-dependencies'])

## 3. `.gitignore`, `.env.example` y secretos

`.gitignore` excluye artefactos recreables, cachés, entornos y secretos. `.env.example` documenta qué variables necesita el proyecto, pero no contiene valores reales. El fichero `.env` es local y no se versiona.

In [ ]:
gitignore = (RAIZ / '.gitignore').read_text()
env_example = (RAIZ / '.env.example').read_text()
print(gitignore)
print(env_example)
assert '.env' in gitignore and '.venv' in gitignore

## 4. JSON, YAML y variables de entorno

Los formatos no compiten: resuelven necesidades distintas.

| Recurso | Uso típico | Ejemplo |
|---|---|---|
| JSON | Parámetros estructurados y versionables | rutas, umbral |
| YAML | Configuración estructurada más legible | perfiles, servicios |
| `.env` | Valores locales o sensibles | token, URL privada |

El proyecto ejecutable usa JSON. YAML y `.env` muestran cómo crecer sin hardcodear valores.

In [ ]:
configuracion = json.loads((RAIZ / 'config/config.json').read_text())
configuracion_yaml = (RAIZ / 'extras/config.yaml').read_text()
api_token = os.getenv('INFORME_API_TOKEN')

print('JSON:', configuracion)
print('YAML de ejemplo:', configuracion_yaml, sep='\n')
print('Token definido:', api_token is not None)

CLAVES_CONFIG = {'input_path', 'output_path', 'importe_minimo'}
def validar_configuracion(contenido):
    ausentes = CLAVES_CONFIG.difference(contenido)
    if ausentes:
        raise ValueError(f'Faltan claves de configuración: {sorted(ausentes)}')
    if float(contenido['importe_minimo']) < 0:
        raise ValueError('importe_minimo no puede ser negativo')

validar_configuracion(configuracion)

## 5. Contrato de datos y pipeline reusable

La validación falla antes de transformar. El pipeline copia la entrada, convierte fechas, filtra operaciones confirmadas, aplica el umbral y agrega por mes, región y canal.

In [ ]:
datos = pd.read_csv(RAIZ / 'data/raw/operaciones.csv')
COLUMNAS = {'fecha', 'region', 'canal', 'estado', 'importe'}

def validar_operaciones(datos):
    ausentes = COLUMNAS.difference(datos.columns)
    if ausentes:
        raise ValueError(f'Faltan columnas obligatorias: {sorted(ausentes)}')
    if datos['importe'].isna().any():
        raise ValueError('importe no puede contener nulos')

def crear_informe(datos, importe_minimo=0.0):
    validar_operaciones(datos)
    salida = datos.copy()
    salida['fecha'] = pd.to_datetime(salida['fecha'], errors='raise')
    salida = salida.loc[
        salida['estado'].eq('confirmada') & salida['importe'].ge(importe_minimo)
    ].copy()
    salida['mes'] = salida['fecha'].dt.to_period('M').astype(str)
    return (salida.groupby(['mes', 'region', 'canal'], as_index=False)
            .agg(importe_total=('importe', 'sum'), operaciones=('importe', 'size'))
            .sort_values(['mes', 'region', 'canal'], ignore_index=True))

informe = crear_informe(datos, configuracion['importe_minimo'])
display(informe)

## 6. Del test manual a una suite de regresión

El patrón original se conserva: observar un fallo, escribir una comprobación, corregir y volver a ejecutar. Una suite combina:

- excepciones para entradas inválidas;
- parametrización para varias reglas equivalentes;
- fixtures para datos reutilizables;
- integración para comprobar el DataFrame completo;
- regresión para proteger una decisión ya corregida;
- efectos secundarios para detectar mutaciones.

In [ ]:
# Test manual y regresión de no mutación
original = datos.copy(deep=True)
resultado = crear_informe(datos, configuracion['importe_minimo'])
assert_frame_equal(datos, original)
assert list(resultado.columns) == ['mes', 'region', 'canal', 'importe_total', 'operaciones']

# La idea de parametrización, ejecutable sin depender de pytest
for umbral, esperado_minimo in [(0, 0), (100, 100), (200, 200)]:
    salida = crear_informe(datos, umbral)
    assert salida['importe_total'].ge(esperado_minimo).all()

print('Comprobaciones manuales superadas.')

### Cómo se formaliza con pytest

```python
@pytest.fixture
def operaciones(): ...

@pytest.mark.parametrize('umbral', [0, 100, 200])
def test_umbral(operaciones, umbral): ...

with pytest.raises(ValueError, match='columnas'):
    validar_operaciones(origen_invalido)
```

La cobertura (`pytest --cov`) indica qué líneas se ejecutaron; no demuestra que los casos sean suficientes.

## 7. Calidad estática y packaging

| Herramienta | Pregunta |
|---|---|
| pytest | ¿Se conserva el comportamiento? |
| Ruff | ¿Hay errores estáticos y convenciones incumplidas? |
| Black | ¿El formato es consistente? |
| mypy | ¿Los tipos declarados son coherentes? |
| packaging | ¿Puede instalarse e importarse como una unidad? |

El script de calidad reúne las comprobaciones locales que después repetirá CI.

In [ ]:
comandos = {
    'tests': 'python -m pytest -q',
    'coverage': 'python -m pytest --cov=informe_operaciones --cov-report=term-missing',
    'ruff': 'python -m ruff check .',
    'black': 'python -m black --check .',
    'mypy': 'python -m mypy src',
    'quality': 'python scripts/check_quality.py',
}
for capa, comando in comandos.items():
    print(f'{capa:10s} -> {comando}')

# Segunda parte — del repositorio al producto operable

## 8. CLI y validación del output

La CLI define una interfaz estable: recibe la ruta de configuración, ejecuta el pipeline y persiste el resultado. `argparse` separa parámetros de terminal de la lógica de negocio.

```bash
python -m informe_operaciones.cli --config config/config.json
```

In [ ]:
print((RAIZ / 'src/informe_operaciones/cli.py').read_text())
print('Salida esperada:', configuracion['output_path'])

## 9. Automatización local: Makefile y script Python

Makefile proporciona targets legibles; el script Python es la alternativa multiplataforma. Ambos reutilizan comandos existentes.

In [ ]:
print((RAIZ / 'Makefile').read_text())
print((RAIZ / 'scripts/check_quality.py').read_text())

## 10. API: publicar la lógica sin duplicarla

Primero probamos una función de consulta. FastAPI solo la expone mediante HTTP. Así el filtro es testeable incluso sin arrancar un servidor.

In [ ]:
def recuperar_informe(informe, region=None):
    respuesta = informe if region is None else informe.loc[informe['region'].eq(region)]
    return respuesta.to_dict(orient='records')

respuesta_norte = recuperar_informe(informe, 'norte')
assert all(fila['region'] == 'norte' for fila in respuesta_norte)
print(respuesta_norte[:2])

try:
    from fastapi import FastAPI
    app = FastAPI(title='Informe de operaciones')

    @app.get('/informe/{region}')
    def consultar_region(region: str):
        return recuperar_informe(informe, region)
    print('Endpoint preparado: GET /informe/{region}')
except ImportError:
    print('FastAPI es opcional; la función de consulta ya está validada.')

## 11. Git, README y CI/CD

Git registra cambios del código; `.gitignore` evita ruido y secretos. El README debe permitir repetir instalación, ejecución y verificación. CI no inventa controles nuevos: instala el proyecto en un entorno limpio y repite el script de calidad. CD solo tiene sentido cuando existe una decisión explícita de despliegue.

In [ ]:
print((RAIZ / 'extras/ci_quality.yml').read_text())
pasos_ci = ['checkout', 'configurar Python', 'instalar .[dev]', 'ejecutar calidad']
assert pasos_ci[-1] == 'ejecutar calidad'
print('Flujo CI:', ' → '.join(pasos_ci))

### Qué hace CI y cuándo empieza CD

```text
cambio → push / pull request → instalar entorno limpio → tests + Ruff + Black + mypy
                                              │
                                      evidencia para aceptar o corregir
                                              │
                              CD: desplegar solo si producto lo autoriza
```

CI responde «¿podemos integrar este cambio con confianza?». CD responde «¿debemos publicar esta versión?». El segundo paso exige una decisión de producto, riesgo y operación; no se activa por tener un workflow.

# Práctica individual — consolidar el mismo proyecto

Los ejercicios siguen el orden de la práctica conjunta. Ninguno introduce una herramienta nueva: cada tarea modifica un dato, una condición o una decisión que ya se ha ejecutado. Las pistas dejan trabajo por completar.

## Ejercicio 1 — Integridad de la estructura

**Tarea.** Crea una copia de `presentes`, elimina `tests/test_pipeline.py` y calcula qué detectaría la validación.

**Entrega.** Entrega la lista con el fichero ausente.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
copia_presentes = presentes.copy()
copia_presentes.remove(...)
ausentes = REQUERIDOS.difference(...)
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 2 — Dependencias declaradas

**Tarea.** Consulta `pyproject` y separa dependencias de ejecución y de desarrollo.

**Entrega.** Entrega dos listas y explica por qué Pandas no debería estar solo en `dev`.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
ejecucion = pyproject['project'][...]
desarrollo = pyproject['project']['optional-dependencies'][...]
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 3 — Configuración inválida

**Tarea.** Crea una copia de `configuracion` sin `output_path` y comprueba que `validar_configuracion` falla.

**Entrega.** Entrega el mensaje de error.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
config_invalida = configuracion.copy()
config_invalida.pop(...)
# Llama a la validación dentro de try/except.
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 4 — Secreto y repositorio

**Tarea.** Decide dónde guardarías `API_TOKEN`, localiza `.env` en `.gitignore` y explica por qué `.env.example` sí puede compartirse.

**Entrega.** Entrega una frase técnica y la línea relevante de `.gitignore`.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
lineas_env = [linea for linea in gitignore.splitlines() if '.env' in linea]
print(...)
# Explica qué contiene cada fichero.
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 5 — Contrato de columnas

**Tarea.** Elimina `canal` de una copia del origen y captura el error de `validar_operaciones`.

**Entrega.** Entrega el error que se produce antes del groupby.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
sin_canal = datos.drop(columns=...)
try:
    validar_operaciones(...)
except ValueError as error:
    ...
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 6 — Umbral de negocio

**Tarea.** Compara el informe con umbral 0 y con umbral 200.

**Entrega.** Entrega número de operaciones e importe total de ambos.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
sin_umbral = crear_informe(datos, ...)
con_umbral = crear_informe(datos, ...)
# Resume con agg o sum.
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 7 — Test parametrizado

**Tarea.** Comprueba con un bucle los umbrales 0, 100 y 200; después escribe cómo sería el decorador de pytest.

**Entrega.** Entrega tres comprobaciones y la cabecera del test parametrizado.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
for umbral in [...]:
    resultado = crear_informe(...)
    assert ...
# @pytest.mark.parametrize(...)
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 8 — Efecto secundario

**Tarea.** Comprueba que `crear_informe` no modifica el DataFrame de entrada.

**Entrega.** Entrega una comparación con `assert_frame_equal`.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
original = datos.copy(deep=True)
_ = crear_informe(...)
assert_frame_equal(..., ...)
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 9 — Test de importes nulos

**Tarea.** Añade a `tests/test_pipeline.py` un test que verifique que un importe nulo hace fallar `validar_operaciones` con un error claro.

**Entrega.** Entrega el nuevo test y ejecuta `python -m pytest -q`.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
datos_invalidos = operaciones.copy()
datos_invalidos.loc[..., 'importe'] = ...
with pytest.raises(ValueError, match=...):
    validar_operaciones(...)
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 10 — Añadir una opción a la CLI

**Tarea.** Añade el flag `--mostrar-ruta` a `cli.py`. Cuando esté presente, debe imprimir la ruta de salida configurada sin ejecutar el pipeline.

**Entrega.** Entrega dos llamadas: una normal y otra con `--mostrar-ruta`.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
parser.add_argument('--mostrar-ruta', action='store_true')
# Después de cargar la configuración, decide si imprimir y terminar.
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 11 — Automatización local

**Tarea.** Ejecuta `make quality` si está disponible. Si no lo está, ejecuta `python scripts/check_quality.py`. Después ejecuta el target o comando que genera el informe.

**Entrega.** Entrega la salida de calidad y confirma si se ha creado el CSV.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
# Prueba make quality.
# Si no existe make, usa el script Python.
# Después ejecuta make pipeline o la CLI.
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 12 — Consulta mediante API

**Tarea.** Pide el informe de la región `sur` usando `recuperar_informe` y consulta una región inexistente.

**Entrega.** Entrega ambas respuestas e indica si cumplen el comportamiento esperado.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
sur = recuperar_informe(informe, ...)
vacia = recuperar_informe(informe, ...)
# Imprime ambas respuestas y explica el resultado.
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 13 — CI sin infraestructura

**Tarea.** Lee `ci_quality.yml` e identifica instalación, tests/calidad y evento que dispara el flujo.

**Entrega.** Entrega una tabla de tres filas.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
contenido_ci = (RAIZ / 'extras/ci_quality.yml').read_text()
# Busca las líneas que contienen run: y on:.
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Ejercicio 14 — README reproducible

**Tarea.** Completa la plantilla de README de la carpeta del proyecto. Debe explicar instalación, ejecución, verificación y el contrato de salida.

**Entrega.** Entrega un README que permita repetir el recorrido desde un entorno limpio.

<details><summary><strong>Obtener pista de ayuda</strong></summary>

~~~python
# Conserva los encabezados existentes.
# Completa «Contrato de salida»: columnas, ruta del CSV, filtro y errores.
# Añade una sección breve de configuración.
~~~
</details>

In [ ]:
# Escribe aquí tu solución.

## Preguntas abiertas

1. ¿Qué capa aporta más valor inmediato en vuestro entorno de trabajo: configuración, tests, CLI o automatización?
2. ¿Qué riesgo asumiríais al desplegar automáticamente este proyecto?
3. ¿Qué contrato añadiríais antes de conectar una fuente de datos real?
4. ¿Qué parte debería revisar obligatoriamente otra persona antes de aceptar el proyecto?